# SP-05 · Conservative Risk-Aware Policy
**Project:** Personalized AI Trading Agent · **Artifact:** `conservative_policy.pt` · **Notebook:** `05_conservative_policy.ipynb`

> **Research and paper-trading use only.** Nothing in this notebook is investment advice, and nothing here guarantees capital protection or drawdown limits. All numbers are produced by running the cells; none are pre-filled.

## What this notebook builds
A GRU/LSTM policy network that *proposes* one of four actions (`0=HOLD, 1=BUY, 2=SELL, 3=EXIT`) with calibrated-on-proxy-labels probabilities and a confidence estimate. It is deliberately biased toward capital preservation: it should prefer HOLD when evidence is weak, avoid initiating positions under prohibited risk conditions, and reduce or close exposure when loss limits are approached.

## Three things you should keep separate while reading results
1. **The learned policy** (`ConservativePolicy`): a neural network that imitates a documented rule. It has no hard guarantees.
2. **The deterministic risk gate** (`RiskGate`): plain Python/NumPy rules with *final authority* over execution. The neural network cannot override it. Every backtest is reported **with and without** the gate so the gate's contribution is visible.
3. **The labels**: in this notebook all labels are **rule-generated proxy labels** (`label_source = "rule_generated_proxy"`), *not* genuine human trader actions. Classification metrics therefore measure how well the network imitates the rule, **not** whether the actions are profitable or safe. Trading performance is judged only by the closed-loop backtest.

## Notebook map
1. Install · 2. Config · 3. Data · 4. Causal features + shared schema · 5. Portfolio simulator, label rule, risk gate · 6. Proxy-label generation · 7. Chronological splits and training-only preprocessing · 8. Model, conservative objective, `ConservativePolicy` · 9. Training with early stopping · 10. Calibration and classification metrics · 11. Closed-loop backtests with costs · 12. Stress tests · 13. Failure analysis · 14. Artifact export and reload check · 15. Assumptions, limitations, Meta-RL contract

In [ ]:
# Colab ships with torch, numpy, pandas, scikit-learn and matplotlib. Only yfinance needs installing.
%pip install -q -U yfinance

In [ ]:
import os, sys, json, math, time, random, hashlib, pickle, platform, warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ----------------------------------------------------------------------------------------------
# QUICK_RUN = True shrinks universe and epochs for a fast smoke test. Use False for real results.
# ----------------------------------------------------------------------------------------------
QUICK_RUN = False

HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTIONS = ("HOLD", "BUY", "SELL", "EXIT")

CONFIG = {
    "policy_id": "SP-05",
    "policy_name": "Conservative Risk-Aware Policy",
    "schema_version": "1.0",
    "label_source": "rule_generated_proxy",     # never "human" in this notebook
    "seed": 42,
    "data": {
        "tickers": ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "WMT", "HD", "UNH"],
        "benchmark": "SPY",
        "start": "2010-01-01",
        "end": None,                            # None = latest available
        "cache_dir": "sp05_data_cache",
        "refresh": False,
    },
    "split": {"train": 0.60, "val": 0.20, "test": 0.20},   # chronological
    "window": 20,                                # sequence length fed to the GRU/LSTM
    # Simulated trader risk-tolerance profiles. These are SYNTHETIC parameters, not real trader data.
    "profiles": [0.7, 1.0, 1.3],
    "eval_risk_tolerance": 1.0,
    "execution": {
        "transaction_cost_bps": 5.0, "slippage_bps": 5.0,   # charged per unit of |exposure change|
        "exposure_step": 0.5, "max_exposure": 1.0,
    },
    # Proxy-label rule thresholds (documented in the labeling section). Hand-set, NOT optimised.
    "label_rule": {
        "entry_vol_pctile_max": 0.70, "entry_trend_min": 0.5, "add_trend_min": 1.0,
        "entry_rsi_range": [0.45, 0.68], "entry_mdd60_floor": -0.15,
        "stop_loss_pct": -0.06, "portfolio_dd_limit": -0.08, "recent_loss_limit": -0.05,
        "reduce_vol_pctile": 0.85, "exit_vol_pctile": 0.95,
        "cooldown_days": 5, "min_hold_days": 3,
    },
    # Deterministic risk gate. Absolute limits, independent of the neural network and of the label rule.
    "risk_gate": {
        "hard_stop_pct": -0.08, "halt_drawdown": -0.10, "block_entry_drawdown": -0.06,
        "block_entry_vol_pctile": 0.90, "force_exit_vol_pctile": 0.98,
        "cooldown_days": 3, "max_exposure": 1.0,
    },
    # Conservative training objective: CE + lambda_cost * E_p[C[y, a]]  (see model section)
    "objective": {
        "lambda_cost": 1.0,
        "lambda_missed_return": 0.5,      # cost of under-exposure relative to the label
        "lambda_excess_exposure": 2.0,    # cost of over-exposure relative to the label (drawdown proxy)
        "lambda_turnover": 0.5,           # cost of trading when the label says HOLD
        "class_weight_power": 0.5,
    },
    "model": {"cell": "gru", "hidden": 64, "layers": 2, "dropout": 0.2},
    "train": {"epochs": 60, "batch_size": 256, "lr": 1e-3, "weight_decay": 1e-4,
              "patience": 8, "grad_clip": 1.0},
    "policy": {"entry_confidence_min": 0.50},   # BUY proposals below this calibrated confidence become HOLD
    "stress": {
        "high_vol_bench_pctile": 0.80, "severe_dd_bench_threshold": -0.07, "min_obs": 10,
        "historical_windows": {
            "2018Q4_selloff": ["2018-10-01", "2018-12-31"],
            "2020_covid_crash": ["2020-02-19", "2020-04-30"],
            "2022_H1_bear": ["2022-01-03", "2022-06-30"],
        },
    },
}

if QUICK_RUN:
    CONFIG["data"]["tickers"] = CONFIG["data"]["tickers"][:4]
    CONFIG["train"]["epochs"] = 3
    CONFIG["train"]["patience"] = 2

WINDOW = CONFIG["window"]
ARTIFACT_DIR = Path("sp05_artifacts"); ARTIFACT_DIR.mkdir(exist_ok=True)
(ARTIFACT_DIR / "plots").mkdir(exist_ok=True)

def set_reproducible(seed):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_reproducible(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"python {platform.python_version()} | torch {torch.__version__} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | sklearn {sklearn.__version__} | device {DEVICE}")

## 3 · Data
Daily OHLCV for a configurable set of liquid US stocks plus a benchmark, downloaded with `yfinance` using **split/dividend-adjusted** prices (`auto_adjust=True`). Files are cached as CSV so re-runs are repeatable; set `CONFIG["data"]["refresh"] = True` to re-download.

Dates are aligned with an **inner join** across all tickers and the benchmark. Tickers are treated as independent single-asset "sleeves" with equal capital. There is no survivorship-bias correction: the default tickers are well-known large caps that survived to today, which flatters buy-and-hold. Treat absolute returns accordingly.

In [ ]:
CACHE_DIR = Path(CONFIG["data"]["cache_dir"]); CACHE_DIR.mkdir(exist_ok=True)
OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _download(ticker, start, end, retries=3):
    import yfinance as yf
    last = None
    for k in range(retries):
        try:
            df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if len(df) > 0:
                return df
        except Exception as e:
            last = e
        time.sleep(2 * (k + 1))
    raise RuntimeError(f"Download failed for {ticker}: {last}")

def load_ohlcv(ticker):
    f = CACHE_DIR / f"{ticker}.csv"
    if f.exists() and not CONFIG["data"]["refresh"]:
        df = pd.read_csv(f, index_col=0, parse_dates=True)
    else:
        df = _download(ticker, CONFIG["data"]["start"], CONFIG["data"]["end"])
        df = df[OHLCV_COLS]
        df.index = pd.to_datetime(df.index).tz_localize(None)
        df.to_csv(f)
    df = df[OHLCV_COLS].sort_index()
    df = df[~df.index.duplicated(keep="last")].dropna()
    bad = (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1) | (df["High"] < df["Low"])
    if bad.any():
        print(f"  {ticker}: dropping {int(bad.sum())} invalid rows")
        df = df[~bad]
    return df

TICKERS = list(CONFIG["data"]["tickers"]); BENCH = CONFIG["data"]["benchmark"]
raw = {t: load_ohlcv(t) for t in TICKERS + [BENCH]}
common = None
for t, d in raw.items():
    common = d.index if common is None else common.intersection(d.index)
common = common.sort_values()
raw = {t: d.loc[common] for t, d in raw.items()}
print(f"{len(TICKERS)} tickers + {BENCH}; {len(common)} common trading days "
      f"({common[0].date()} -> {common[-1].date()})")
assert len(common) > 1500, "Need at least ~6 years of common history for features, splits and stress tests."
pd.DataFrame({t: [d.index[0].date(), len(d)] for t, d in raw.items()}, index=["first_date", "rows"]).T

## 4 · Causal features and the shared schema
Every feature at decision time *t* uses data up to and including the close of day *t* only (rolling windows look backward; no centered windows, no future fills). The decision is made at the close of *t*; the resulting exposure earns the **close-to-close return of day t+1**, and costs are charged on that return. Nothing from *t+1* enters any feature.

| Group | Features |
|---|---|
| Market (per ticker) | `ret_1d, ret_5d, ret_20d, ret_60d, rv_20, atr_pct_14, vol_pctile_252, mdd_60, dd_from_peak_60, gap, gap_z, abnormal_move, vol_anom, trend_strength, dist_sma50, rsi_14, bb_width_20` |
| Market context (benchmark) | `bench_ret_20d, bench_vol_pctile, bench_dd_60` |
| Position / account state | `exposure, position_size, unrealized_pnl, holding_days_norm, recent_loss_10d, portfolio_drawdown_60d, days_since_exit_norm` |
| Trader | `trader_risk_tolerance, trader_activity_20d` |

Full definitions are in `FEATURE_DEFS` in the next cell and are exported verbatim to `feature_schema.json`.

**Schema note.** The task refers to a "shared schema" used by the other four policy notebooks. I do not have those notebooks, so this notebook defines schema version `1.0` explicitly (names, order, units, definitions, window length, action map) and writes it to `feature_schema.json` with a SHA-256 hash. If your other notebooks use different names or ordering, compare `feature_schema.json` files and map them before integration.

In [ ]:
FEATURE_DEFS = {
 # name: (group, unit, definition)
 "ret_1d": ("market", "fraction", "Close-to-close simple return of day t."),
 "ret_5d": ("market", "fraction", "5-day simple return ending at t."),
 "ret_20d": ("market", "fraction", "20-day simple return ending at t (momentum)."),
 "ret_60d": ("market", "fraction", "60-day simple return ending at t (slower momentum)."),
 "rv_20": ("market", "annualised vol", "Std of 20 daily log returns x sqrt(252)."),
 "atr_pct_14": ("market", "fraction of price", "14-day simple-average true range divided by close."),
 "vol_pctile_252": ("market", "0..1", "Percentile rank of rv_20 within its own trailing 252 values (includes t)."),
 "mdd_60": ("market", "fraction (<=0)", "Rolling maximum drawdown: worst peak-to-trough decline inside the trailing 60 closes."),
 "dd_from_peak_60": ("market", "fraction (<=0)", "Close / trailing-60-day max close - 1."),
 "gap": ("market", "fraction", "Open(t) / Close(t-1) - 1."),
 "gap_z": ("market", "z-score", "gap divided by trailing 60-day std of gaps."),
 "abnormal_move": ("market", "0/1", "1 if |ret_1d| exceeds 3 x the trailing 20-day daily-return std measured up to t-1."),
 "vol_anom": ("market", "z-score", "z-score of log1p(volume) vs its trailing 20-day mean/std."),
 "trend_strength": ("market", "ATR units", "(SMA20 - SMA50) / ATR14."),
 "dist_sma50": ("market", "fraction", "Close / SMA50 - 1."),
 "rsi_14": ("market", "0..1", "Wilder RSI(14) divided by 100."),
 "bb_width_20": ("market", "fraction", "Bollinger Band width (upper-lower)/middle with 20 days, 2 std."),
 "bench_ret_20d": ("context", "fraction", "20-day return of the benchmark."),
 "bench_vol_pctile": ("context", "0..1", "vol_pctile_252 of the benchmark."),
 "bench_dd_60": ("context", "fraction (<=0)", "Benchmark close / trailing-60-day max - 1."),
 "exposure": ("state", "0..max_exposure", "Current fraction of the sleeve's capital invested (before this decision)."),
 "position_size": ("state", "equity multiples", "exposure x current sleeve equity (notional relative to starting capital)."),
 "unrealized_pnl": ("state", "fraction", "Close / volume-weighted entry price - 1 while invested, else 0."),
 "holding_days_norm": ("state", "0..1", "Decision-days since entry, capped at 60, divided by 60."),
 "recent_loss_10d": ("state", "fraction (<=0)", "min(0, sum of the sleeve's last 10 daily strategy returns)."),
 "portfolio_drawdown_60d": ("state", "fraction (<=0)", "Sleeve equity / max sleeve equity over trailing 60 days - 1 (rolling window so a flat sleeve can recover)."),
 "days_since_exit_norm": ("state", "0..1", "Days since the last full exit, capped at 20, divided by 20 (1.0 while invested or never exited)."),
 "trader_risk_tolerance": ("trader", "multiplier", "Scales loss limits and entry volatility ceiling. SYNTHETIC profile (0.7 / 1.0 / 1.3)."),
 "trader_activity_20d": ("trader", "0..1", "Fraction of the last 20 decisions that were non-HOLD (behavioural footprint)."),
}
MKT_FEATURES = [k for k, v in FEATURE_DEFS.items() if v[0] == "market"]
CTX_FEATURES = [k for k, v in FEATURE_DEFS.items() if v[0] == "context"]
STATE_FEATURES = [k for k, v in FEATURE_DEFS.items() if v[0] == "state"]
TRADER_FEATURES = [k for k, v in FEATURE_DEFS.items() if v[0] == "trader"]
ALL_FEATURES = MKT_FEATURES + CTX_FEATURES + STATE_FEATURES + TRADER_FEATURES
NF = len(ALL_FEATURES); FIDX = {n: i for i, n in enumerate(ALL_FEATURES)}
N_MKT = len(MKT_FEATURES) + len(CTX_FEATURES)          # columns that come from market data
SINCE_EXIT_CAP = 20
SCHEMA_HASH = hashlib.sha256(json.dumps(ALL_FEATURES).encode()).hexdigest()

def _rsi(close, n=14):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    rsi = 100 - 100 / (1 + up / dn.replace(0, np.nan))
    return rsi.fillna(100.0).where(up.notna())

def market_features(df):
    c, o, h, l, v = df["Close"], df["Open"], df["High"], df["Low"], df["Volume"]
    r = c.pct_change(); lr = np.log(c).diff(); prev = c.shift(1)
    tr = pd.concat([h - l, (h - prev).abs(), (l - prev).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    rv20 = lr.rolling(20).std() * np.sqrt(252)
    f = pd.DataFrame(index=df.index)
    f["ret_1d"] = r; f["ret_5d"] = c.pct_change(5); f["ret_20d"] = c.pct_change(20); f["ret_60d"] = c.pct_change(60)
    f["rv_20"] = rv20
    f["atr_pct_14"] = atr / c
    f["vol_pctile_252"] = rv20.rolling(252, min_periods=252).rank(pct=True)
    f["mdd_60"] = c.rolling(60).apply(lambda x: float(np.min(x / np.maximum.accumulate(x) - 1.0)), raw=True)
    f["dd_from_peak_60"] = c / c.rolling(60).max() - 1.0
    gap = o / prev - 1.0
    f["gap"] = gap; f["gap_z"] = gap / gap.rolling(60).std()
    sig_prev = lr.rolling(20).std().shift(1)
    f["abnormal_move"] = ((r.abs() / sig_prev) > 3.0).astype(float).where(sig_prev.notna())
    lv = np.log1p(v)
    f["vol_anom"] = (lv - lv.rolling(20).mean()) / lv.rolling(20).std()
    sma20, sma50 = c.rolling(20).mean(), c.rolling(50).mean()
    f["trend_strength"] = (sma20 - sma50) / atr
    f["dist_sma50"] = c / sma50 - 1.0
    f["rsi_14"] = _rsi(c) / 100.0
    f["bb_width_20"] = 4.0 * c.rolling(20).std() / sma20
    return f.replace([np.inf, -np.inf], np.nan)

feat_by_ticker = {t: market_features(raw[t]) for t in TICKERS}
bf = market_features(raw[BENCH])
ctx = pd.DataFrame({"bench_ret_20d": bf["ret_20d"], "bench_vol_pctile": bf["vol_pctile_252"],
                    "bench_dd_60": bf["dd_from_peak_60"]})

# Panel arrays: PX (T,N) closes, MKT (T,N,N_MKT) market+context features, RET_NEXT (T,N) return of t -> t+1.
close_df = pd.DataFrame({t: raw[t]["Close"] for t in TICKERS})
MKT_full = np.stack([pd.concat([feat_by_ticker[t][MKT_FEATURES], ctx], axis=1).to_numpy() for t in TICKERS], axis=1)
valid = ~np.isnan(MKT_full).any(axis=(1, 2))
first_valid = int(np.argmax(valid))
assert valid[first_valid:].mean() > 0.999, "Unexpected NaNs after warm-up; inspect the data."

DATES = close_df.index[first_valid:]
PX = close_df.to_numpy()[first_valid:]
MKT = np.nan_to_num(MKT_full[first_valid:], nan=0.0).astype(np.float64)
BENCH_PX = raw[BENCH]["Close"].to_numpy()[first_valid:]
RET_NEXT = np.vstack([PX[1:] / PX[:-1] - 1.0, np.full((1, len(TICKERS)), np.nan)])
BENCH_RET_NEXT = np.concatenate([BENCH_PX[1:] / BENCH_PX[:-1] - 1.0, [np.nan]])
T_TOTAL = len(DATES); N_DEC = T_TOTAL - 1          # decisions possible at t = 0 .. T-2
print(f"Usable panel: {T_TOTAL} days after warm-up ({DATES[0].date()} -> {DATES[-1].date()}), "
      f"{NF} features per step ({N_MKT} from market data)")
assert set(FEATURE_DEFS) == set(ALL_FEATURES) and len(ALL_FEATURES) == len(set(ALL_FEATURES))

## 5 · Labels: a documented, explicit, causal risk rule
**Label provenance: `rule_generated_proxy`.** No genuine human trader actions are used here. The network is trained to imitate the rule below, so its ceiling is "behaves like this rule", and any bias or blind spot in the rule is inherited.

All rule inputs are available at the close of day *t*. The rule never looks at future prices, so it is **not** a hindsight-optimal exit/entry oracle, and it will take losses (stops trigger *after* losses occur).

Exposure moves in steps of 0.5 up to a maximum of 1.0. Actions: **BUY** adds 0.5, **SELL** reduces by 0.5, **EXIT** goes to zero, **HOLD** changes nothing. Priority order, evaluated per sleeve at each close (limits scale by the trader's `risk_tolerance` *r*):

1. **EXIT (hard risk breach)** while invested: unrealized P&L <= `stop_loss_pct`·r, **or** 60-day account drawdown <= `portfolio_dd_limit`·r, **or** an abnormal *down* move (>3σ), **or** volatility percentile >= `exit_vol_pctile` with negative trend.
2. **SELL/EXIT (soft de-risk)** after the minimum holding period: volatility percentile > `reduce_vol_pctile`, **or** 10-day account loss <= `recent_loss_limit`·r, **or** clearly negative trend and momentum. Exposure 1.0 → SELL (halve); exposure 0.5 → EXIT.
3. **BUY (new entry)** only if *no prohibition* applies (volatility percentile <= `entry_vol_pctile_max`·r, rolling max-drawdown above the floor, account not beyond half its drawdown limit, no recent-loss breach, no abnormal move today, cooldown since last exit respected) **and** evidence is strong (trend strength, positive 20d momentum, RSI inside a non-extreme band, price above SMA50). Scaling in from 0.5 to 1.0 additionally requires a profit, minimum holding period and a lower volatility ceiling.
4. **HOLD** otherwise: weak evidence and no limit breached means do nothing.

### How the trade-off between returns, drawdown and turnover is defined
* **In the labels:** thresholds are hand-set, not fitted. Strong-evidence entry conditions and entry prohibitions trade return opportunity against drawdown risk; cooldown and minimum-hold periods limit turnover.
* **In the training objective** (model section): a cost matrix `C[y, a]` penalises the network's action `a` relative to the label `y` using three explicit weights: `lambda_missed_return` (under-exposure vs label), `lambda_excess_exposure` (over-exposure vs label, a drawdown proxy, weighted 4x higher by default) and `lambda_turnover` (trading when the label is HOLD). These weights are **not** calibrated to realised P&L and encode a preference, not a measured optimum. Change them in `CONFIG["objective"]`.

In [ ]:
I = FIDX.__getitem__
EXEC = CONFIG["execution"]; MAX_EXP = EXEC["max_exposure"]

class PortfolioTracker:
    """Vectorised account simulator: one independent single-asset sleeve per episode."""
    def __init__(self, n, exec_cfg):
        self.n = n; self.step = exec_cfg["exposure_step"]; self.max_exp = exec_cfg["max_exposure"]
        self.cost = (exec_cfg["transaction_cost_bps"] + exec_cfg["slippage_bps"]) / 1e4
        self.exposure = np.zeros(n); self.entry_px = np.full(n, np.nan); self.hold = np.zeros(n)
        self.since_exit = np.full(n, float(SINCE_EXIT_CAP))
        self.equity = np.ones(n); self.eq_hist = np.ones((n, 60))
        self.ret_hist = np.zeros((n, 10)); self.act_hist = np.zeros((n, 20))

    def state(self, px, risk_tol):
        """State + trader features (after market features in schema order). Uses only past/current close."""
        entry = np.where(np.isnan(self.entry_px), px, self.entry_px)
        unreal = np.where(self.exposure > 0, px / entry - 1.0, 0.0)
        dd = self.equity / self.eq_hist.max(axis=1) - 1.0
        return np.column_stack([
            self.exposure, self.exposure * self.equity, unreal, np.minimum(self.hold, 60) / 60.0,
            np.minimum(self.ret_hist.sum(axis=1), 0.0), dd,
            np.minimum(self.since_exit, SINCE_EXIT_CAP) / SINCE_EXIT_CAP,
            risk_tol, self.act_hist.mean(axis=1)])

    def sanitize(self, a):
        a = np.asarray(a).copy()
        a[((a == SELL) | (a == EXIT)) & (self.exposure <= 1e-9)] = HOLD
        a[(a == BUY) & (self.exposure >= self.max_exp - 1e-9)] = HOLD
        return a

    def apply(self, a, px):
        old = self.exposure.copy(); new = old.copy()
        new = np.where(a == BUY, np.minimum(old + self.step, self.max_exp), new)
        new = np.where(a == SELL, np.maximum(old - self.step, 0.0), new)
        new = np.where(a == EXIT, 0.0, new)
        prev_entry = np.where(np.isnan(self.entry_px), 0.0, self.entry_px)
        wavg = (old * prev_entry + (new - old) * px) / np.where(new > 0, new, 1.0)
        self.entry_px = np.where(new > old, wavg, self.entry_px)
        self.entry_px = np.where(new <= 0, np.nan, self.entry_px)
        closed = (new <= 0) & (old > 0)
        self.hold = np.where(new > 0, np.where(old > 0, self.hold + 1, 1.0), 0.0)
        self.since_exit = np.where(closed, 0.0, np.where(new > 0, float(SINCE_EXIT_CAP),
                                                         np.minimum(self.since_exit + 1, SINCE_EXIT_CAP)))
        self.exposure = new
        self.act_hist = np.column_stack([self.act_hist[:, 1:], (a != HOLD).astype(float)])
        return np.abs(new - old)

    def advance(self, ret_next, turnover):
        pnl = self.exposure * ret_next - self.cost * turnover
        self.equity = self.equity * (1.0 + pnl)
        self.eq_hist = np.column_stack([self.eq_hist[:, 1:], self.equity])
        self.ret_hist = np.column_stack([self.ret_hist[:, 1:], pnl])
        return pnl

def _g(x, name): return x[:, FIDX[name]]

def teacher_rule(x, cfg=None):
    """Explicit, causal, risk-aware proxy-label rule. x: (E, NF) raw features at decision time t."""
    c = cfg or CONFIG["label_rule"]
    r = _g(x, "trader_risk_tolerance"); exp = _g(x, "exposure"); unreal = _g(x, "unrealized_pnl")
    dd = _g(x, "portfolio_drawdown_60d"); rl = _g(x, "recent_loss_10d")
    vp = _g(x, "vol_pctile_252"); tr = _g(x, "trend_strength"); mom = _g(x, "ret_20d")
    rsi = _g(x, "rsi_14"); d50 = _g(x, "dist_sma50"); mdd = _g(x, "mdd_60")
    ab = _g(x, "abnormal_move") > 0.5; r1 = _g(x, "ret_1d")
    hold = _g(x, "holding_days_norm") * 60.0; since = _g(x, "days_since_exit_norm") * SINCE_EXIT_CAP
    stop, dd_lim, rl_lim = c["stop_loss_pct"] * r, c["portfolio_dd_limit"] * r, c["recent_loss_limit"] * r
    vol_max = np.minimum(c["entry_vol_pctile_max"] * r, 0.95)
    long_ = exp > 1e-9
    hard = long_ & ((unreal <= stop) | (dd <= dd_lim) | (ab & (r1 < 0)) | ((vp >= c["exit_vol_pctile"]) & (tr < 0)))
    soft = (long_ & ~hard & (hold >= c["min_hold_days"]) &
            ((vp > c["reduce_vol_pctile"]) | (rl <= rl_lim) | ((tr < -0.5) & (mom < 0))))
    prohibited = ((vp > vol_max) | (mdd < c["entry_mdd60_floor"]) | (dd <= 0.5 * dd_lim) |
                  (rl <= rl_lim) | ab | (since < c["cooldown_days"]))
    lo, hi = c["entry_rsi_range"]
    strong = (tr >= c["entry_trend_min"]) & (mom > 0) & (rsi >= lo) & (rsi <= hi) & (d50 > 0)
    buy_new = (exp <= 1e-9) & ~prohibited & strong
    scale_in = (long_ & (exp < MAX_EXP - 1e-9) & ~hard & ~soft & ~prohibited & (tr >= c["add_trend_min"]) &
                (unreal > 0) & (hold >= c["min_hold_days"]) & (vp <= 0.8 * vol_max))
    a = np.zeros(len(x), dtype=int)
    a[buy_new | scale_in] = BUY
    a[soft & (exp > 0.5 + 1e-9)] = SELL
    a[soft & (exp <= 0.5 + 1e-9)] = EXIT
    a[hard] = EXIT
    return a

class RiskGate:
    """Deterministic risk controller with FINAL authority. Pure NumPy rules; no learned components.
    Hard-stop / halt rules may force EXIT; entry rules may block BUY. It never creates a BUY."""
    REASONS = {0: "none", 1: "infeasible_action", 2: "forced_exit_hard_stop", 3: "forced_exit_drawdown_halt",
               4: "forced_exit_extreme_vol_downtrend", 5: "blocked_entry_high_vol", 6: "blocked_entry_drawdown",
               7: "blocked_entry_abnormal_move", 8: "blocked_entry_cooldown"}
    def __init__(self, cfg=None):
        self.c = cfg or CONFIG["risk_gate"]

    def apply(self, proposed, x):
        c = self.c; a = np.asarray(proposed).copy(); reason = np.zeros(len(a), dtype=int)
        exp = _g(x, "exposure"); unreal = _g(x, "unrealized_pnl"); dd = _g(x, "portfolio_drawdown_60d")
        vp = _g(x, "vol_pctile_252"); tr = _g(x, "trend_strength"); ab = _g(x, "abnormal_move") > 0.5
        since = _g(x, "days_since_exit_norm") * SINCE_EXIT_CAP
        infeasible = (((a == SELL) | (a == EXIT)) & (exp <= 1e-9)) | ((a == BUY) & (exp >= c["max_exposure"] - 1e-9))
        a[infeasible] = HOLD; reason[infeasible] = 1
        long_ = exp > 1e-9
        for mask, code in [(long_ & (unreal <= c["hard_stop_pct"]), 2), (long_ & (dd <= c["halt_drawdown"]), 3),
                           (long_ & (vp >= c["force_exit_vol_pctile"]) & (tr < 0), 4)]:
            hit = mask & (a != EXIT) & (reason == 0)
            a[hit] = EXIT; reason[hit] = code
        buy = a == BUY
        for mask, code in [(vp > c["block_entry_vol_pctile"], 5), (dd <= c["block_entry_drawdown"], 6),
                           (ab, 7), ((since < c["cooldown_days"]) & (exp <= 1e-9), 8)]:
            hit = buy & mask & (reason == 0)
            a[hit] = HOLD; reason[hit] = code
        return a, reason

def decide_rule_controller(x, buf):
    """Independent, simple baseline: trend filter + volatility cap + hard stop. Not the label rule."""
    vp = _g(x, "vol_pctile_252"); d50 = _g(x, "dist_sma50"); tr = _g(x, "trend_strength")
    exp = _g(x, "exposure"); unreal = _g(x, "unrealized_pnl"); since = _g(x, "days_since_exit_norm") * SINCE_EXIT_CAP
    want = (d50 > 0) & (tr > 0) & (vp < 0.80)
    a = np.zeros(len(x), dtype=int)
    a[want & (exp < MAX_EXP - 1e-9) & (since >= 5)] = BUY
    a[(~want | (unreal <= -0.06)) & (exp > 0)] = EXIT
    return a, None

def decide_teacher(x, buf):
    return teacher_rule(x), None
print("Tracker, label rule and risk gate defined. Gate reasons:", RiskGate.REASONS)

In [ ]:
def run_episodes(decide, start, end, tick_idx, risk_tol, record_from=None, gate=None, collect_features=False):
    """Closed-loop simulation over decision times t in [start, end).
    - Features at t come from data up to close(t) plus the simulated account state.
    - The final action sets exposure at close(t); that exposure earns RET_NEXT[t] (close t -> close t+1).
    - Steps before `record_from` are forced HOLD warm-up so sequence windows are full (nothing is traded)."""
    tick_idx = np.asarray(tick_idx); E = len(tick_idx); risk_tol = np.asarray(risk_tol, dtype=float)
    record_from = start if record_from is None else record_from
    assert end <= N_DEC and start >= 0
    trk = PortfolioTracker(E, EXEC); buf = np.zeros((E, WINDOW, NF), dtype=np.float32)
    rec = {k: [] for k in ["t", "proposed", "final", "reason", "ret", "exposure", "turnover", "conf", "probs", "x"]}
    for t in range(start, end):
        st = trk.state(PX[t][tick_idx], risk_tol)
        x = np.concatenate([MKT[t][tick_idx], st], axis=1)
        buf[:, :-1] = buf[:, 1:]; buf[:, -1] = x
        if t < record_from:
            proposed, aux = np.zeros(E, dtype=int), None
        else:
            proposed, aux = decide(x, buf)
        if gate is not None:
            final, reason = gate.apply(proposed, x)
        else:
            final, reason = proposed.copy(), np.zeros(E, dtype=int)
        final = trk.sanitize(final)
        turnover = trk.apply(final, PX[t][tick_idx])
        pnl = trk.advance(RET_NEXT[t][tick_idx], turnover)
        if t >= record_from:
            rec["t"].append(t); rec["proposed"].append(proposed); rec["final"].append(final)
            rec["reason"].append(reason); rec["ret"].append(pnl); rec["exposure"].append(trk.exposure.copy())
            rec["turnover"].append(turnover)
            if aux is not None:
                rec["conf"].append(aux["confidence"]); rec["probs"].append(aux["probs"])
            if collect_features:
                rec["x"].append(x.astype(np.float32))
    out = {k: np.array(v) for k, v in rec.items() if len(v) > 0}
    out["tick_idx"] = tick_idx
    return out

# ---- Generate proxy labels: roll the rule forward on every (ticker, risk-profile) episode -------------
PROFILES = CONFIG["profiles"]
EP_TICK = np.repeat(np.arange(len(TICKERS)), len(PROFILES))
EP_RISK = np.tile(np.array(PROFILES), len(TICKERS))
E_ALL = len(EP_TICK)
teacher_roll = run_episodes(decide_teacher, 0, N_DEC, EP_TICK, EP_RISK, gate=None, collect_features=True)
FEATS_RAW = teacher_roll["x"]                 # (N_DEC, E, NF), float32
LABELS = teacher_roll["proposed"].astype(np.int64)   # (N_DEC, E), rule-generated PROXY labels
assert FEATS_RAW.shape == (N_DEC, E_ALL, NF) and LABELS.shape == (N_DEC, E_ALL)
assert np.isfinite(FEATS_RAW).all()

dist = pd.Series(LABELS.ravel()).map(dict(enumerate(ACTIONS))).value_counts()
print("Proxy-label distribution (all episodes, all dates):")
print((dist / dist.sum()).round(4).to_string())
assert set(np.unique(LABELS)).issubset({0, 1, 2, 3}), "Unexpected label values"
if (dist / dist.sum()).get("BUY", 0) < 0.002:
    print("WARNING: very few BUY labels. Loosen CONFIG['label_rule'] entry thresholds or use more tickers/history.")

# Causality self-check: truncating the future must not change features or labels at earlier times.
cut = N_DEC // 2
chk = run_episodes(decide_teacher, 0, cut, EP_TICK, EP_RISK, gate=None, collect_features=True)
assert np.array_equal(chk["proposed"], LABELS[:cut]) and np.allclose(chk["x"], FEATS_RAW[:cut]), "Look-ahead detected!"
print("Causality check passed: labels/features before t do not change when future data is removed.")

## 6 · Chronological splits and training-only preprocessing
Decision times are split **chronologically**: first 60% train, next 20% validation, final 20% test. No shuffling across time. The validation set drives early stopping and temperature scaling; the **test set is used only for final reporting**.

* The `StandardScaler` is fit on **training-period rows only** and then applied unchanged to validation/test and at inference. Scaled values are clipped to ±5.
* Sequence windows (20 steps) for the first validation/test samples reach back into the preceding split. They contain only *past features*, not future information. Labels depend only on information at *t*, so no label leaks across the boundary.
* Account-state features are produced by the rule-driven simulator over each split continuously. During the **closed-loop backtests** later, the state instead comes from the *model's own* (gated) actions, which is a distribution shift from the training data; this matters and is discussed in the limitations.

In [ ]:
TRAIN_END = int(N_DEC * CONFIG["split"]["train"])
VAL_END = int(N_DEC * (CONFIG["split"]["train"] + CONFIG["split"]["val"]))
SPLITS = {"train": (WINDOW - 1, TRAIN_END), "val": (TRAIN_END, VAL_END), "test": (VAL_END, N_DEC)}
assert all(hi - lo > 100 for lo, hi in SPLITS.values()), "A split is too short; use more history."
def date_of(t): return DATES[t].date()
print("Decision-time splits (chronological):")
for k, (lo, hi) in SPLITS.items():
    print(f"  {k:5s}: t={lo:5d}..{hi - 1:5d}  {date_of(lo)} -> {date_of(hi - 1)}  ({hi - lo} days x {E_ALL} episodes)")

scaler = StandardScaler().fit(FEATS_RAW[:TRAIN_END].reshape(-1, NF))      # TRAINING ROWS ONLY
CLIP = 5.0
FEATS_SCALED = np.clip((FEATS_RAW - scaler.mean_.astype(np.float32)) / scaler.scale_.astype(np.float32),
                       -CLIP, CLIP).astype(np.float32)
FEATS_T = torch.as_tensor(FEATS_SCALED, device=DEVICE)
LABELS_T = torch.as_tensor(LABELS, device=DEVICE)

def split_index(name, shuffle=False, seed=0):
    lo, hi = SPLITS[name]
    tt = np.repeat(np.arange(lo, hi), E_ALL); ee = np.tile(np.arange(E_ALL), hi - lo)
    if shuffle:
        p = np.random.default_rng(seed).permutation(len(tt)); tt, ee = tt[p], ee[p]
    return tt, ee

def get_batch(tt, ee):
    ti = torch.as_tensor(tt, device=DEVICE)[:, None] + torch.arange(-WINDOW + 1, 1, device=DEVICE)[None, :]
    ei = torch.as_tensor(ee, device=DEVICE)[:, None]
    return FEATS_T[ti, ei], LABELS_T[torch.as_tensor(tt, device=DEVICE), torch.as_tensor(ee, device=DEVICE)]

lab_train = LABELS[SPLITS["train"][0]:SPLITS["train"][1]].ravel()
class_counts = np.bincount(lab_train, minlength=4).astype(float)
print("Train label counts:", dict(zip(ACTIONS, class_counts.astype(int))))

## 7 · Model, conservative objective and `ConservativePolicy`
**Network.** Per-step `LayerNorm` → 2-layer GRU (or LSTM; `CONFIG["model"]["cell"]`) → MLP head → four logits. Output probabilities are `softmax(logits / T)` where `T` is a temperature fitted on the validation set. Two confidence numbers are returned: `confidence` = max calibrated probability and `confidence_entropy` = `1 - H(p)/ln 4`.

**Conservative objective.** `loss = weighted_CE + lambda_cost * E_{a~p}[C[y, a]]`, with `C[y,a] = λ_missed·max(0, Δ(y)−Δ(a)) + λ_excess·max(0, Δ(a)−Δ(y)) + λ_turn·1[y=HOLD ∧ a≠HOLD]` and exposure deltas `Δ = {HOLD:0, BUY:+0.5, SELL:−0.5, EXIT:−1.0}`. Defaults make over-exposure errors (predicting BUY when the label says SELL/EXIT, or HOLD when it says EXIT) four times costlier than missed-upside errors, and add a small penalty for unnecessary trading. Class weights (inverse frequency to the power 0.5) counter the dominant HOLD class.

**What this does not do.** The objective biases the network toward *proxy-label* conservatism. It does not make the network safe, and it is not tuned to realised drawdown or P&L. A BUY proposal whose calibrated confidence is below `entry_confidence_min` is converted to HOLD inside `predict` (a policy-level preference to HOLD when evidence is weak); SELL/EXIT are never filtered. Hard safety constraints live exclusively in `RiskGate`.

**Confidence calibration limitations.** Temperature scaling is fit on validation *proxy labels*, so "calibrated" means agreement with the rule's labels in that period, **not** the probability that an action is profitable or safe. Calibration drifts when market regimes change, is not guaranteed on the test period or live, ignores class-conditional miscalibration, and Expected Calibration Error depends on binning and sample size. Treat `confidence` as a relative ranking signal, not a probability of success.

In [ ]:
class ConservativeGRUNet(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.2, n_actions=4, cell="gru"):
        super().__init__()
        self.cfg = dict(n_features=n_features, hidden=hidden, layers=layers, dropout=dropout,
                        n_actions=n_actions, cell=cell)
        self.norm = nn.LayerNorm(n_features)
        rnn_cls = nn.GRU if cell == "gru" else nn.LSTM
        self.rnn = rnn_cls(n_features, hidden, num_layers=layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_actions))
    def forward(self, x):
        out, _ = self.rnn(self.norm(x))
        return self.head(out[:, -1])

def build_cost_matrix(obj):
    delta = np.array([0.0, 0.5, -0.5, -1.0])
    C = np.zeros((4, 4))
    for y in range(4):
        for a in range(4):
            missed = max(0.0, delta[y] - delta[a]); excess = max(0.0, delta[a] - delta[y])
            turn = 1.0 if (y == HOLD and a != HOLD) else 0.0
            C[y, a] = obj["lambda_missed_return"] * missed + obj["lambda_excess_exposure"] * excess + obj["lambda_turnover"] * turn
    return C

class ConservativeObjective(nn.Module):
    def __init__(self, class_weights, cost_matrix, lambda_cost):
        super().__init__()
        self.register_buffer("cw", torch.as_tensor(class_weights, dtype=torch.float32))
        self.register_buffer("C", torch.as_tensor(cost_matrix, dtype=torch.float32))
        self.lam = float(lambda_cost)
    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, weight=self.cw)
        exp_cost = (F.softmax(logits, dim=1) * self.C[y]).sum(dim=1).mean()
        return ce + self.lam * exp_cost

def action_entropy_confidence(p):
    h = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(axis=-1)
    return 1.0 - h / math.log(p.shape[-1])

class ConservativePolicy:
    """Proposes actions. It has NO authority to execute: the deterministic RiskGate decides what executes."""
    ACTIONS = ACTIONS
    FORMAT_VERSION = 1

    def __init__(self, model, feature_names, window, mean, scale, clip=5.0, temperature=1.0,
                 entry_confidence_min=0.5, device="cpu", meta=None):
        self.device = torch.device(device)
        self.model = model.to(self.device).eval()
        self.feature_names = list(feature_names); self.window = int(window)
        self.mean = np.asarray(mean, dtype=np.float32); self.scale = np.asarray(scale, dtype=np.float32)
        self.clip = float(clip); self.temperature = float(temperature)
        self.entry_confidence_min = float(entry_confidence_min); self.meta = dict(meta or {})

    def _prepare(self, state):
        if isinstance(state, pd.DataFrame):
            missing = [c for c in self.feature_names if c not in state.columns]
            if missing: raise ValueError(f"Missing features: {missing}")
            state = state[self.feature_names].to_numpy()
        if isinstance(state, torch.Tensor): state = state.detach().cpu().numpy()
        arr = np.asarray(state, dtype=np.float32)
        single = arr.ndim == 2
        if single: arr = arr[None]
        if arr.ndim != 3 or arr.shape[2] != len(self.feature_names):
            raise ValueError(f"Expected (L,{len(self.feature_names)}) or (B,L,{len(self.feature_names)}); got {arr.shape}")
        if arr.shape[1] < self.window: raise ValueError(f"Need at least {self.window} time steps; got {arr.shape[1]}")
        arr = arr[:, -self.window:]
        if not np.isfinite(arr).all(): raise ValueError("State contains NaN/inf; refusing to predict.")
        return np.clip((arr - self.mean) / self.scale, -self.clip, self.clip), single

    @torch.no_grad()
    def predict_arrays(self, state):
        """state: RAW (un-scaled) windows (L,F) or (B,L,F) in schema order. Returns dict of arrays."""
        x, single = self._prepare(state)
        logits = self.model(torch.as_tensor(x, device=self.device)).cpu().numpy().astype(np.float64)
        z = logits / self.temperature; z -= z.max(axis=1, keepdims=True)
        p = np.exp(z); p /= p.sum(axis=1, keepdims=True)
        raw_action = p.argmax(axis=1); conf = p.max(axis=1)
        action = raw_action.copy()
        filtered = (action == BUY) & (conf < self.entry_confidence_min)
        action[filtered] = HOLD
        return {"probs": p, "raw_action": raw_action, "action": action, "confidence": conf,
                "confidence_entropy": action_entropy_confidence(p), "confidence_filtered": filtered, "single": single}

    def predict(self, state):
        """Returns dict (single window) or list of dicts (batch): action probabilities, selected action, confidence."""
        o = self.predict_arrays(state)
        res = [{"action_probabilities": {n: float(o["probs"][i, k]) for k, n in enumerate(ACTIONS)},
                "action": int(o["action"][i]), "action_name": ACTIONS[int(o["action"][i])],
                "raw_model_action": int(o["raw_action"][i]), "confidence": float(o["confidence"][i]),
                "confidence_entropy": float(o["confidence_entropy"][i]),
                "entry_filtered_to_hold": bool(o["confidence_filtered"][i]),
                "is_proposal_only": True} for i in range(len(o["action"]))]
        return res[0] if o["single"] else res

    def save(self, path):
        torch.save({"format_version": self.FORMAT_VERSION,
                    "state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
                    "model_config": dict(self.model.cfg), "feature_names": self.feature_names, "window": self.window,
                    "scaler_mean": torch.as_tensor(self.mean), "scaler_scale": torch.as_tensor(self.scale),
                    "clip": self.clip, "temperature": self.temperature,
                    "entry_confidence_min": self.entry_confidence_min, "actions": list(ACTIONS),
                    "meta": self.meta}, str(path))

    @classmethod
    def load(cls, path, device="cpu"):
        ck = torch.load(str(path), map_location=device, weights_only=True)
        model = ConservativeGRUNet(**ck["model_config"]); model.load_state_dict(ck["state_dict"])
        return cls(model, ck["feature_names"], ck["window"], ck["scaler_mean"].numpy(), ck["scaler_scale"].numpy(),
                   ck["clip"], ck["temperature"], ck["entry_confidence_min"], device, ck["meta"])

COST_MATRIX = build_cost_matrix(CONFIG["objective"])
print("Cost matrix C[label, predicted] (rows HOLD,BUY,SELL,EXIT):")
print(pd.DataFrame(COST_MATRIX, index=ACTIONS, columns=ACTIONS).round(2))

In [ ]:
set_reproducible(CONFIG["seed"])
mc, tc = CONFIG["model"], CONFIG["train"]
model = ConservativeGRUNet(NF, mc["hidden"], mc["layers"], mc["dropout"], 4, mc["cell"]).to(DEVICE)
cw = (class_counts.sum() / np.maximum(class_counts, 1.0) / 4.0) ** CONFIG["objective"]["class_weight_power"]
objective = ConservativeObjective(cw / cw.mean(), COST_MATRIX, CONFIG["objective"]["lambda_cost"]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)

@torch.no_grad()
def collect_logits(name, bs=2048):
    model.eval(); tt, ee = split_index(name); L, Y = [], []
    for i in range(0, len(tt), bs):
        x, y = get_batch(tt[i:i + bs], ee[i:i + bs]); L.append(model(x)); Y.append(y)
    return torch.cat(L), torch.cat(Y)

history, best = [], {"val_loss": float("inf"), "epoch": -1, "state": None}
bad_epochs = 0; t0 = time.time()
for epoch in range(tc["epochs"]):
    model.train(); tt, ee = split_index("train", shuffle=True, seed=CONFIG["seed"] + epoch)
    run, nb = 0.0, 0
    for i in range(0, len(tt), tc["batch_size"]):
        x, y = get_batch(tt[i:i + tc["batch_size"]], ee[i:i + tc["batch_size"]])
        opt.zero_grad(set_to_none=True); loss = objective(model(x), y); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), tc["grad_clip"]); opt.step()
        run += loss.item(); nb += 1
    vl, vy = collect_logits("val"); val_loss = objective(vl, vy).item()
    val_acc = (vl.argmax(1) == vy).float().mean().item(); sched.step(val_loss)
    history.append({"epoch": epoch, "train_loss": run / nb, "val_loss": val_loss, "val_acc": val_acc,
                    "lr": opt.param_groups[0]["lr"]})
    improved = val_loss < best["val_loss"] - 1e-5
    if improved:
        best.update(val_loss=val_loss, epoch=epoch, state={k: v.detach().clone() for k, v in model.state_dict().items()})
        bad_epochs = 0
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | train {run / nb:.4f} | val {val_loss:.4f} | val_acc {val_acc:.3f}"
          f"{' *' if improved else ''} | {time.time() - t0:5.0f}s")
    if bad_epochs >= tc["patience"]:
        print(f"Early stopping at epoch {epoch}; best epoch {best['epoch']}"); break
model.load_state_dict(best["state"]); model.eval()
hist_df = pd.DataFrame(history)
print(f"Restored best weights from epoch {best['epoch']} (val objective {best['val_loss']:.4f})")

In [ ]:
# ---- Temperature scaling on VALIDATION proxy labels (never on test) -----------------------------------
vl, vy = collect_logits("val")
log_T = torch.zeros(1, device=DEVICE, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_T], lr=0.1, max_iter=100)
def _closure():
    lbfgs.zero_grad(); l = F.cross_entropy(vl / log_T.exp(), vy); l.backward(); return l
lbfgs.step(_closure)
TEMPERATURE = float(log_T.exp().clamp(0.25, 10.0).item())
print(f"Fitted temperature T = {TEMPERATURE:.3f} (T>1 means the raw network was overconfident on validation)")

def ece_score(probs, labels, bins=15):
    conf = probs.max(1); correct = (probs.argmax(1) == labels).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0; rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
            rows.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    return float(e), rows

policy = ConservativePolicy(model, ALL_FEATURES, WINDOW, scaler.mean_, scaler.scale_, CLIP, TEMPERATURE,
                            CONFIG["policy"]["entry_confidence_min"], str(DEVICE),
                            meta={"policy_id": CONFIG["policy_id"], "schema_version": CONFIG["schema_version"],
                                  "schema_hash": SCHEMA_HASH, "label_source": CONFIG["label_source"],
                                  "best_epoch": int(best["epoch"]), "seed": CONFIG["seed"],
                                  "research_paper_trading_only": True})

# ---- Classification metrics vs PROXY labels (imitation fidelity, not trading quality) -----------------
def class_metrics(name):
    lg, y = collect_logits(name); y = y.cpu().numpy()
    p_raw = F.softmax(lg, 1).cpu().numpy(); p_cal = F.softmax(lg / TEMPERATURE, 1).cpu().numpy()
    pred = p_cal.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1, 2, 3], zero_division=0)
    ece_raw, _ = ece_score(p_raw, y); ece_cal, rel = ece_score(p_cal, y)
    return {"n": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=[0, 1, 2, 3], zero_division=0)),
            "majority_class_accuracy": float(np.bincount(y, minlength=4).max() / len(y)),
            "per_class": {ACTIONS[k]: {"precision": float(pr[k]), "recall": float(rc[k]), "f1": float(f1[k]),
                                        "support": int(sup[k])} for k in range(4)},
            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1, 2, 3]).tolist(),
            "ece_uncalibrated": ece_raw, "ece_temperature_scaled": ece_cal,
            "mean_confidence": float(p_cal.max(1).mean()), "_reliability": rel,
            "risk_increasing_errors": {  # predicted more exposure than the label asked for
                "pred_BUY_when_label_SELL_or_EXIT": float(((pred == BUY) & ((y == SELL) | (y == EXIT))).sum() / max(1, ((y == SELL) | (y == EXIT)).sum())),
                "pred_HOLD_when_label_EXIT": float(((pred == HOLD) & (y == EXIT)).sum() / max(1, (y == EXIT).sum()))}}

CLS = {k: class_metrics(k) for k in ["val", "test"]}
for k in ["val", "test"]:
    m = CLS[k]
    print(f"[{k}] acc {m['accuracy']:.3f} (majority-class {m['majority_class_accuracy']:.3f}) | "
          f"bal.acc {m['balanced_accuracy']:.3f} | macro-F1 {m['macro_f1']:.3f} | "
          f"ECE raw {m['ece_uncalibrated']:.3f} -> scaled {m['ece_temperature_scaled']:.3f}")
print("\nTest per-class vs PROXY labels:")
print(pd.DataFrame(CLS["test"]["per_class"]).T.round(3))
print("Risk-increasing disagreement rates (test):", {k: round(v, 3) for k, v in CLS["test"]["risk_increasing_errors"].items()})

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
ax[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train"); ax[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val")
ax[0].axvline(best["epoch"], ls="--", c="gray"); ax[0].set_title("Conservative objective"); ax[0].set_xlabel("epoch"); ax[0].legend()
cm = np.array(CLS["test"]["confusion_matrix"]); ax[1].imshow(cm, cmap="Blues")
for i in range(4):
    for j in range(4): ax[1].text(j, i, cm[i, j], ha="center", va="center", color="black" if cm[i, j] < cm.max() / 2 else "white")
ax[1].set_xticks(range(4)); ax[1].set_xticklabels(ACTIONS); ax[1].set_yticks(range(4)); ax[1].set_yticklabels(ACTIONS)
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("proxy label"); ax[1].set_title("Test confusion matrix (vs rule labels)")
rel = CLS["test"]["_reliability"]
if rel:
    ax[2].plot([0, 1], [0, 1], "k--", lw=1); ax[2].scatter([r[0] for r in rel], [r[1] for r in rel], s=[8 + r[2] / 50 for r in rel])
ax[2].set_xlabel("mean confidence"); ax[2].set_ylabel("accuracy vs proxy labels"); ax[2].set_title("Reliability (test, proxy labels)")
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "plots" / "training_and_classification.png", dpi=120); plt.show()

## 8 · Closed-loop backtest on the held-out test period
Unlike the classification metrics above, this evaluates **trading behaviour**. Each strategy trades every ticker as an independent sleeve starting flat, with the sleeves combined as a daily-rebalanced equal-weight portfolio.

* **Timing:** decision at the close of *t* → exposure earns the *t→t+1* close-to-close return. A 10 bps cost (5 bps commission/fees + 5 bps slippage) is charged per unit of exposure change.
* **State:** account-state features (exposure, unrealized P&L, drawdown, ...) are generated by *each strategy's own* executed actions, so the network sees states produced by its own behaviour, not the teacher's.
* **Strategies compared**
  * `model_gated`: learned policy → deterministic `RiskGate` → execution (the intended deployment configuration)
  * `model_ungated`: learned policy alone, to isolate what the gate adds
  * `teacher_rule`: the proxy-label rule itself (the thing the network imitates)
  * `rule_controller`: a separate simple risk controller (SMA50 trend + volatility cap + 6% stop)
  * `buy_hold`: equal-weight buy-and-hold of the same tickers; `benchmark_buy_hold`: the benchmark ETF
* Reported: cumulative return, Sharpe (rf = 0), Sortino, max drawdown, VaR/CVaR, downside deviation, exposure and turnover.

Results are for one specific period and one training seed. They carry no statistical significance claim.

In [ ]:
EVAL_E = len(TICKERS); EVAL_TICK = np.arange(EVAL_E); EVAL_RISK = np.full(EVAL_E, CONFIG["eval_risk_tolerance"])
COST_RATE = (EXEC["transaction_cost_bps"] + EXEC["slippage_bps"]) / 1e4
GATE = RiskGate()

def decide_model(x, buf):
    o = policy.predict_arrays(buf)
    return o["action"].astype(int), {"confidence": o["confidence"], "probs": o["probs"]}

def buy_hold(start, end, tick_idx):
    ret = RET_NEXT[start:end][:, tick_idx].copy(); ret[0] -= COST_RATE * MAX_EXP
    turn = np.zeros_like(ret); turn[0] = MAX_EXP
    return {"t": np.arange(start, end), "ret": ret, "exposure": np.full_like(ret, MAX_EXP), "turnover": turn, "tick_idx": tick_idx}

def bench_hold(start, end):
    ret = BENCH_RET_NEXT[start:end][:, None].copy(); ret[0] -= COST_RATE * MAX_EXP
    turn = np.zeros_like(ret); turn[0] = MAX_EXP
    return {"t": np.arange(start, end), "ret": ret, "exposure": np.full_like(ret, MAX_EXP), "turnover": turn, "tick_idx": None}

def run_strategies(start, end):
    warm = start - (WINDOW - 1); assert warm >= 0
    kw = dict(tick_idx=EVAL_TICK, risk_tol=EVAL_RISK, record_from=start)
    return {
        "model_gated": run_episodes(decide_model, warm, end, gate=GATE, **kw),
        "model_ungated": run_episodes(decide_model, warm, end, gate=None, **kw),
        "teacher_rule": run_episodes(decide_teacher, warm, end, gate=None, **kw),
        "rule_controller": run_episodes(decide_rule_controller, warm, end, gate=None, **kw),
        "buy_hold": buy_hold(start, end, EVAL_TICK),
        "benchmark_buy_hold": bench_hold(start, end)}

def perf_metrics(r, periods=252):
    r = np.asarray(r, dtype=float); n = len(r)
    eq = np.cumprod(1 + r); peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]; dd = eq / peak - 1
    sd = r.std(ddof=1) if n > 1 else np.nan; yrs = n / periods
    dsd = np.sqrt(np.mean(np.minimum(r, 0) ** 2)) * np.sqrt(periods)
    var5 = np.percentile(r, 5); k = min(20, n - 1)
    roll20 = (eq[k:] / np.concatenate([[1.0], eq[:-1]])[:len(eq) - k] - 1).min() if n > 20 else np.nan
    cagr = eq[-1] ** (1 / yrs) - 1 if yrs > 0 and eq[-1] > 0 else np.nan
    mdd = dd.min()
    return {"n_days": n, "cumulative_return": float(eq[-1] - 1), "cagr": float(cagr),
            "ann_volatility": float(sd * np.sqrt(periods)),
            "sharpe": float(r.mean() / sd * np.sqrt(periods)) if sd and sd > 0 else None,
            "sortino": float(r.mean() * periods / dsd) if dsd > 0 else None,
            "max_drawdown": float(mdd), "calmar": float(cagr / abs(mdd)) if mdd < 0 else None,
            "ulcer_index": float(np.sqrt(np.mean(dd ** 2))), "downside_deviation": float(dsd),
            "var_5pct_daily": float(var5), "cvar_5pct_daily": float(r[r <= var5].mean()),
            "worst_day": float(r.min()), "worst_20d": float(roll20) if roll20 == roll20 else None,
            "positive_day_rate": float((r > 0).mean())}

def summarize(res):
    r = res["ret"].mean(axis=1); m = perf_metrics(r); yrs = len(r) / 252
    m["avg_exposure"] = float(res["exposure"].mean()); m["pct_days_invested"] = float((res["exposure"] > 0).mean())
    m["annualized_turnover"] = float(res["turnover"].sum(axis=0).mean() / yrs)
    m["trades_per_year_per_ticker"] = float((res["turnover"] > 0).sum(axis=0).mean() / yrs)
    return m

TS, TE = SPLITS["test"]
BT = run_strategies(TS, TE)
BT_METRICS = {k: summarize(v) for k, v in BT.items()}
show_cols = ["cumulative_return", "cagr", "ann_volatility", "sharpe", "sortino", "max_drawdown", "calmar",
             "cvar_5pct_daily", "downside_deviation", "avg_exposure", "annualized_turnover", "pct_days_invested"]
print(f"TEST period {date_of(TS)} -> {date_of(TE - 1)} | cost {COST_RATE * 1e4:.0f} bps per unit exposure change\n")
metrics_table = pd.DataFrame(BT_METRICS).T[show_cols]
print(metrics_table.round(4).to_string())

# What does the gate contribute, and what did the learned policy propose?
g = BT["model_gated"]; u = BT["model_ungated"]
REASON_NAMES = RiskGate.REASONS
gate_stats = {"steps": int(g["final"].size),
              "proposals_by_action": {ACTIONS[k]: int((g["proposed"] == k).sum()) for k in range(4)},
              "final_actions_by_action": {ACTIONS[k]: int((g["final"] == k).sum()) for k in range(4)},
              "gate_interventions_by_reason": {REASON_NAMES[k]: int((g["reason"] == k).sum()) for k in range(1, 9)},
              "gate_override_rate_excluding_infeasible": float(((g["reason"] > 1)).mean()),
              "ungated_proposals_by_action": {ACTIONS[k]: int((u["proposed"] == k).sum()) for k in range(4)},
              "ungated_infeasible_proposals": int((u["proposed"] != u["final"]).sum())}
print("\nLearned-policy proposals vs gate:"); print(json.dumps(gate_stats, indent=1))

pt = []
for j, tk in enumerate(TICKERS):
    mg = perf_metrics(g["ret"][:, j]); mb = perf_metrics(BT["buy_hold"]["ret"][:, j])
    pt.append({"ticker": tk, "model_gated_ret": mg["cumulative_return"], "buy_hold_ret": mb["cumulative_return"],
               "model_gated_mdd": mg["max_drawdown"], "buy_hold_mdd": mb["max_drawdown"],
               "model_avg_exposure": float(g["exposure"][:, j].mean())})
per_ticker = pd.DataFrame(pt).set_index("ticker"); print("\nPer-ticker (test, gated model vs buy-and-hold):")
print(per_ticker.round(3).to_string())

# Equity curves aligned to the day each return is earned (t+1)
EQ_DATES = DATES[BT["model_gated"]["t"] + 1]
eq_df = pd.DataFrame({"date": EQ_DATES, "split": "test"})
for k, v in BT.items():
    eq_df[f"equity_{k}"] = np.cumprod(1 + v["ret"].mean(axis=1))
eq_df["avg_exposure_model_gated"] = g["exposure"].mean(axis=1)
eq_df["avg_exposure_model_ungated"] = u["exposure"].mean(axis=1)
eq_df["gate_interventions_model_gated"] = (g["reason"] > 1).sum(axis=1)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 9))
names = {"model_gated": "Model + risk gate", "model_ungated": "Model only", "teacher_rule": "Teacher rule",
         "rule_controller": "Simple risk controller", "buy_hold": "Buy & hold (EW)", "benchmark_buy_hold": f"{BENCH} buy & hold"}
for k, lab in names.items():
    eq = eq_df[f"equity_{k}"].to_numpy(); dd = eq / np.maximum.accumulate(eq) - 1
    ax[0, 0].plot(eq_df["date"], eq, label=lab, lw=2 if k == "model_gated" else 1.2)
    ax[0, 1].plot(eq_df["date"], dd, label=lab, lw=2 if k == "model_gated" else 1.2)
ax[0, 0].set_title("Cumulative growth of 1.0 (test, after costs)"); ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=.3)
ax[0, 1].set_title("Drawdown"); ax[0, 1].grid(alpha=.3)
ax[1, 0].plot(eq_df["date"], eq_df["avg_exposure_model_gated"], label="model + gate")
ax[1, 0].plot(eq_df["date"], eq_df["avg_exposure_model_ungated"], label="model only", alpha=.7)
bv = MKT[BT["model_gated"]["t"], 0, FIDX["bench_vol_pctile"]]
ax[1, 0].fill_between(eq_df["date"], 0, 1, where=bv >= CONFIG["stress"]["high_vol_bench_pctile"], color="red", alpha=.08, label="benchmark high-vol")
ax[1, 0].set_title("Average exposure across tickers"); ax[1, 0].set_ylim(0, 1.05); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=.3)
gi = {k: v for k, v in gate_stats["gate_interventions_by_reason"].items() if v > 0 and k != "infeasible_action"}
if gi: ax[1, 1].barh(list(gi.keys()), list(gi.values()))
else: ax[1, 1].text(0.5, 0.5, "No gate interventions in test", ha="center", va="center")
ax[1, 1].set_title("Risk-gate interventions (excl. infeasible), test")
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "plots" / "backtest.png", dpi=120); plt.show()

In [ ]:
SC = CONFIG["stress"]; STRAT_KEYS = ["model_gated", "model_ungated", "teacher_rule", "rule_controller", "buy_hold"]

def subset_metrics(r, mask, min_obs):
    n = int(mask.sum())
    if n < min_obs: return {"n_days": n, "note": f"insufficient observations (<{min_obs})"}
    rs = r[mask]; eq = np.cumprod(1 + rs); dd = eq / np.maximum.accumulate(eq) - 1; v5 = np.percentile(rs, 5)
    return {"n_days": n, "cum_return_on_selected_days": float(eq[-1] - 1), "mean_daily_return": float(rs.mean()),
            "daily_vol": float(rs.std(ddof=1)), "worst_day": float(rs.min()),
            "cvar_5pct_daily": float(rs[rs <= v5].mean()), "max_drawdown_selected_days_concatenated": float(dd.min())}

# (A) Out-of-sample regime slices inside the TEST period, selected from benchmark conditions known at decision time
tt = BT["model_gated"]["t"]
bvol = MKT[tt, 0, FIDX["bench_vol_pctile"]]; bdd = MKT[tt, 0, FIDX["bench_dd_60"]]
REGIMES = {f"benchmark_vol_pctile >= {SC['high_vol_bench_pctile']}": bvol >= SC["high_vol_bench_pctile"],
           f"benchmark 60d drawdown <= {SC['severe_dd_bench_threshold']}": bdd <= SC["severe_dd_bench_threshold"]}
STRESS = {"test_regime_slices": {}, "historical_windows": {}}
for rname, mask in REGIMES.items():
    STRESS["test_regime_slices"][rname] = {k: subset_metrics(BT[k]["ret"].mean(axis=1), mask, SC["min_obs"]) for k in STRAT_KEYS}
    STRESS["test_regime_slices"][rname]["_days_in_regime"] = int(mask.sum())
print("(A) OUT-OF-SAMPLE regime slices within the test period (non-contiguous days; compounding is illustrative):")
for rname, d in STRESS["test_regime_slices"].items():
    print(f"\n  {rname}  [{d['_days_in_regime']} of {len(tt)} test days]")
    rows = {k: v for k, v in d.items() if not k.startswith("_")}
    print(pd.DataFrame(rows).T.round(4).to_string())

# (B) Named historical stress windows. Most fall in TRAIN/VAL, so they are IN-SAMPLE and optimistic.
def split_status(a, b):
    if b <= TRAIN_END: return "IN-SAMPLE (train period; optimistic)"
    if a >= TRAIN_END and b <= VAL_END: return "validation period (used for early stopping/calibration)"
    if a >= VAL_END: return "out-of-sample (test period)"
    return "straddles splits (partly in-sample)"
print("\n(B) Named historical windows (closed-loop re-run from a flat start at the window start):")
for wname, (ws, we) in SC["historical_windows"].items():
    a = int(np.searchsorted(DATES, pd.Timestamp(ws))); b = min(int(np.searchsorted(DATES, pd.Timestamp(we), side="right")) - 1, N_DEC)
    if a - (WINDOW - 1) < 0 or b - a < 5 or a >= N_DEC:
        print(f"  {wname}: skipped (outside usable data range)"); continue
    res = run_strategies(a, b); status = split_status(a, b)
    STRESS["historical_windows"][wname] = {"dates": [str(date_of(a)), str(date_of(b - 1))], "sample_status": status,
        "strategies": {k: {m: v for m, v in summarize(res[k]).items()
                           if m in ("cumulative_return", "max_drawdown", "cvar_5pct_daily", "worst_day", "avg_exposure", "annualized_turnover")}
                       for k in list(res)}}
    print(f"\n  {wname} {date_of(a)} -> {date_of(b - 1)} | {status}")
    print(pd.DataFrame(STRESS["historical_windows"][wname]["strategies"]).T.round(4).to_string())
if not STRESS["historical_windows"]:
    print("  (none of the configured windows overlap the usable data range)")

In [ ]:
# Where does the model + gate still lose money? Everything below is computed from the test run.
g = BT["model_gated"]; tt = g["t"]; R, E = g["ret"].shape
rows = []
for j in range(E):
    for i in range(R):
        rows.append((i, j, g["ret"][i, j]))
flat = pd.DataFrame(rows, columns=["i", "j", "strat_ret"]).sort_values("strat_ret").head(10)
worst = []
for _, r in flat.iterrows():
    i, j = int(r["i"]), int(r["j"]); t = tt[i]
    worst.append({"date_earned": str(DATES[t + 1].date()), "ticker": TICKERS[j], "strategy_return": float(r["strat_ret"]),
                  "underlying_return": float(RET_NEXT[t, j]), "exposure_held": float(g["exposure"][i, j]),
                  "model_proposed": ACTIONS[int(g["proposed"][i, j])], "executed": ACTIONS[int(g["final"][i, j])],
                  "gate_reason": REASON_NAMES[int(g["reason"][i, j])],
                  "vol_pctile": float(MKT[t, j, FIDX["vol_pctile_252"]]),
                  "model_conf": float(g["conf"][i, j]) if "conf" in g else None})
worst_df = pd.DataFrame(worst); print("10 worst sleeve-days for model + gate (test):"); print(worst_df.round(4).to_string(index=False))

# Loss-avoidance diagnostic: how much exposure was held on days when the underlying fell sharply?
under = RET_NEXT[tt][:, EVAL_TICK]; big_down = under <= -0.03
exp_all = float(g["exposure"].mean())
loss_avoid = {"n_sleeve_days_underlying_down_3pct_or_more": int(big_down.sum()),
              "avg_exposure_on_those_days": float(g["exposure"][big_down].mean()) if big_down.any() else None,
              "avg_exposure_all_days": exp_all,
              "share_of_those_days_fully_flat": float((g["exposure"][big_down] == 0).mean()) if big_down.any() else None,
              "share_of_those_days_at_full_exposure": float((g["exposure"][big_down] >= MAX_EXP - 1e-9).mean()) if big_down.any() else None,
              "note": "Exposure is set at the prior close, so the policy cannot react to the move itself (gap/overnight risk)."}
print("\nLoss-avoidance diagnostic:"); print(json.dumps(loss_avoid, indent=1))

def drawdown_episodes(r, dates, k=3):
    eq = np.cumprod(1 + r); peak = np.maximum.accumulate(eq); dd = eq / peak - 1; eps = []; i = 0; n = len(r)
    while i < n:
        if dd[i] < 0:
            s = i - 1 if i > 0 else 0; j = i
            while j < n and dd[j] < 0: j += 1
            seg = dd[i:j]; tr = i + int(np.argmin(seg))
            eps.append({"peak_date": str(dates[s].date()), "trough_date": str(dates[tr].date()), "depth": float(seg.min()),
                        "recovered": bool(j < n), "start_idx": s, "trough_idx": tr}); i = j
        else: i += 1
    return sorted(eps, key=lambda e: e["depth"])[:k]
r_g = g["ret"].mean(axis=1); r_bh = BT["buy_hold"]["ret"].mean(axis=1)
dd_eps = drawdown_episodes(r_g, EQ_DATES)
for e in dd_eps:
    e["avg_exposure_peak_to_trough"] = float(g["exposure"][e["start_idx"]:e["trough_idx"] + 1].mean())
    bh_seg = np.cumprod(1 + r_bh[e["start_idx"]:e["trough_idx"] + 1]); e["buy_hold_decline_same_span"] = float(bh_seg[-1] / 1.0 - 1)
    e.pop("start_idx"); e.pop("trough_idx")
print("\nLargest portfolio drawdown episodes for model + gate (test):"); print(pd.DataFrame(dd_eps).round(4).to_string(index=False))

mg, mb = BT_METRICS["model_gated"], BT_METRICS["buy_hold"]
FAILURE = {"worst_sleeve_days": worst, "loss_avoidance": loss_avoid, "largest_drawdown_episodes": dd_eps,
           "vs_buy_hold": {"max_drawdown_model": mg["max_drawdown"], "max_drawdown_buy_hold": mb["max_drawdown"],
                           "model_reduced_max_drawdown": bool(mg["max_drawdown"] > mb["max_drawdown"]),
                           "cum_return_model": mg["cumulative_return"], "cum_return_buy_hold": mb["cumulative_return"],
                           "model_underperformed_buy_hold_return": bool(mg["cumulative_return"] < mb["cumulative_return"])},
           "tickers_where_model_drawdown_worse_than_buy_hold": [tk for tk in TICKERS if per_ticker.loc[tk, "model_gated_mdd"] < per_ticker.loc[tk, "buy_hold_mdd"]],
           "gate_vs_ungated": {"max_drawdown_gated": BT_METRICS["model_gated"]["max_drawdown"],
                               "max_drawdown_ungated": BT_METRICS["model_ungated"]["max_drawdown"],
                               "cum_return_gated": BT_METRICS["model_gated"]["cumulative_return"],
                               "cum_return_ungated": BT_METRICS["model_ungated"]["cumulative_return"]}}
print("\nSummary flags (computed, not asserted):"); print(json.dumps(FAILURE["vs_buy_hold"], indent=1))
print("Tickers where the gated model's max drawdown was WORSE than buy-and-hold:", FAILURE["tickers_where_model_drawdown_worse_than_buy_hold"] or "none")
print("\nReminder: drawdowns that remain are losses neither the network nor the gate avoided; stops act only after a loss has occurred "
      "and cannot protect against overnight gaps or sudden regime breaks.")

## 9 · Export artifacts
Writes `conservative_policy.pt`, `conservative_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json` and `equity_curve.csv` to `sp05_artifacts/`. The `.pt` file is self-contained (weights, scaler statistics, feature order, temperature, window) and loads with `torch.load(..., weights_only=True)`, so it does not depend on pickle. `preprocessing.pkl` is a convenience copy of the fitted scikit-learn scaler; only unpickle files you created yourself.

In [ ]:
def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items() if not str(k).startswith("_")}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o or o in (float("inf"), float("-inf"))) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Timestamp, dt.date)): return str(o)
    return o

P = {k: ARTIFACT_DIR / k for k in ["conservative_policy.pt", "conservative_config.json", "feature_schema.json",
                                    "preprocessing.pkl", "evaluation_metrics.json", "equity_curve.csv"]}

# feature_schema.json
schema = {"schema_name": "personalized_ai_trading_agent_state", "schema_version": CONFIG["schema_version"],
          "defined_by": "SP-05 notebook (shared schema not available at authoring time; verify against other policy notebooks)",
          "schema_hash_sha256": SCHEMA_HASH, "window_length": WINDOW, "n_features": NF, "feature_order": ALL_FEATURES,
          "features": [{"name": n, "group": FEATURE_DEFS[n][0], "unit": FEATURE_DEFS[n][1], "definition": FEATURE_DEFS[n][2],
                        "causal": True} for n in ALL_FEATURES],
          "input_tensor": f"float32 array (window_length={WINDOW}, n_features={NF}), RAW un-scaled values, oldest row first",
          "actions": {str(i): a for i, a in enumerate(ACTIONS)},
          "action_semantics": {"HOLD": "no change", "BUY": f"+{EXEC['exposure_step']} exposure up to {MAX_EXP}",
                               "SELL": f"-{EXEC['exposure_step']} exposure", "EXIT": "exposure to 0"},
          "timing": "features use data up to close of t; decision at close t; exposure earns return t -> t+1"}
json.dump(to_py(schema), open(P["feature_schema.json"], "w"), indent=2)

# preprocessing.pkl
pickle.dump({"scaler": scaler, "feature_names": ALL_FEATURES, "clip": CLIP, "window": WINDOW, "schema_hash": SCHEMA_HASH,
             "fit_on": {"split": "train", "first_decision_date": str(date_of(0)), "last_decision_date": str(date_of(TRAIN_END - 1))},
             "sklearn_version": sklearn.__version__}, open(P["preprocessing.pkl"], "wb"))

# conservative_policy.pt
policy.save(P["conservative_policy.pt"])

CONTRACT = {
  "role": "Proposal generator only. The deterministic risk controller/gate has final execution authority.",
  "input": {"type": f"float32 (L={WINDOW}, F={NF}) or (B, L, F), RAW features in feature_schema.json order",
            "state_features_required": STATE_FEATURES + TRADER_FEATURES,
            "note": "Account-state features must be computed by the caller exactly as defined in feature_schema.json."},
  "output": {"action_probabilities": "dict HOLD/BUY/SELL/EXIT (temperature-scaled)", "action": "int 0..3", "action_name": "str",
             "raw_model_action": "argmax before the BUY confidence filter", "confidence": "max calibrated probability",
             "confidence_entropy": "1 - H(p)/ln 4", "entry_filtered_to_hold": "bool", "is_proposal_only": True},
  "confidence_caveat": "Calibrated to rule-generated proxy labels on one validation period; NOT a probability of profit or safety.",
  "meta_rl_usage": ["Use action_probabilities and confidence as input features/priors, not as commands.",
                    "Route every final action through the deterministic risk gate; never bypass it.",
                    "Do not treat this policy's proxy-label accuracy as evidence of trading skill."],
  "compatibility_checks": ["feature_schema.json hash equals the Meta-RL's expected schema hash",
                           "window length and action indices (0=HOLD,1=BUY,2=SELL,3=EXIT) match the other policies"]}

cfg_out = {"config": CONFIG, "resolved": {
    "n_features": NF, "feature_names": ALL_FEATURES, "schema_hash_sha256": SCHEMA_HASH, "temperature": TEMPERATURE,
    "best_epoch": int(best["epoch"]), "epochs_run": len(history), "device": str(DEVICE),
    "data_span": [str(DATES[0].date()), str(DATES[-1].date())], "tickers": TICKERS, "benchmark": BENCH,
    "splits": {k: [str(date_of(lo)), str(date_of(hi - 1))] for k, (lo, hi) in SPLITS.items()},
    "cost_matrix": COST_MATRIX.tolist(), "class_counts_train": dict(zip(ACTIONS, class_counts.tolist())),
    "label_source": CONFIG["label_source"], "human_action_labels_used": False,
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "purpose": "research and paper trading only"},
    "integration_contract": CONTRACT}
json.dump(to_py(cfg_out), open(P["conservative_config.json"], "w"), indent=2)

eval_out = {"label_source": CONFIG["label_source"],
            "interpretation": {"classification": "agreement with rule-generated proxy labels (imitation fidelity), not trading quality",
                               "backtest": "closed-loop, after costs and slippage, single seed, single test period; no significance claim",
                               "gate": "model_gated applies the deterministic RiskGate; model_ungated does not"},
            "test_period": [str(date_of(TS)), str(date_of(TE - 1))], "cost_bps_per_unit_exposure_change": COST_RATE * 1e4,
            "classification": CLS, "temperature": TEMPERATURE, "backtest_test_period": BT_METRICS,
            "gate_statistics": gate_stats, "per_ticker_test": per_ticker.reset_index().to_dict("records"),
            "stress_tests": STRESS, "failure_cases": FAILURE}
json.dump(to_py(eval_out), open(P["evaluation_metrics.json"], "w"), indent=2)
eq_df.to_csv(P["equity_curve.csv"], index=False)
for k, p in P.items(): print(f"saved {p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Reload from disk and verify the exported policy reproduces the in-memory one.
loaded = ConservativePolicy.load(P["conservative_policy.pt"], device="cpu")
sample_t = np.linspace(SPLITS["test"][0], N_DEC - 1, 64).astype(int); sample_e = np.arange(64) % E_ALL
win = np.stack([FEATS_RAW[t - WINDOW + 1:t + 1, e] for t, e in zip(sample_t, sample_e)])      # RAW windows
a = policy.predict_arrays(win); b = loaded.predict_arrays(win)
assert np.allclose(a["probs"], b["probs"], atol=1e-4), "Reload mismatch: exported policy differs from in-memory policy"
one = loaded.predict(win[0]); print("single-window predict():", json.dumps(one, indent=1))
many = loaded.predict(pd.DataFrame(win[0], columns=ALL_FEATURES)); assert many["action"] == one["action"]
assert abs(sum(one["action_probabilities"].values()) - 1) < 1e-6
sch = json.load(open(P["feature_schema.json"])); assert sch["schema_hash_sha256"] == SCHEMA_HASH and sch["feature_order"] == ALL_FEATURES
try: loaded.predict(win[0][:, :-1]); raise SystemError("shape check failed")
except ValueError as e: print("Malformed input correctly rejected:", e)
print("\nExport verification passed.\n")
print("ARTIFACT PATHS"); [print("  ", p.resolve()) for p in P.values()]
print("   plots:", [str(x.resolve()) for x in (ARTIFACT_DIR / "plots").glob("*.png")])
try:
    import shutil; from google.colab import files
    z = shutil.make_archive("sp05_artifacts", "zip", ARTIFACT_DIR); files.download(z)
except Exception:
    print("(Not in Colab or download unavailable: copy the sp05_artifacts/ folder manually.)")

## 10 · Assumptions, limitations and Meta-RL integration contract

### Assumptions
* **Labels are rule-generated proxies**, not human actions (`label_source = "rule_generated_proxy"`). Classification scores measure imitation of a hand-set rule.
* Thresholds, cost-matrix weights, profile multipliers (0.7/1.0/1.3) and the gate limits are **hand-set a priori**, not optimised on test data and not proven suitable for any real trader.
* Each ticker is an independent sleeve with equal capital; "portfolio drawdown" in the state is that sleeve's own rolling-60-day equity drawdown. Long-only, fractional exposure, daily close execution, flat 10 bps all-in cost, no borrowing, taxes, spreads that widen under stress, or liquidity limits.
* `yfinance` adjusted prices are used as-is; no survivorship-bias or data-vendor error correction.
* The shared schema is defined here (`feature_schema.json`, v1.0) because the other four notebooks were not available.

### Limitations (read before trusting any number)
* **No guarantee of capital protection.** Stops and the gate act after losses begin and cannot prevent overnight gap losses, trading halts or fast regime breaks. Report sections 8, 12 and 13 show where losses still occurred.
* **Distribution shift:** the network trains on states produced by the rule's trajectories but is evaluated closed-loop on states produced by its own actions.
* **Single training seed, single test window, ten large caps:** no statistical-significance claim; results may not transfer to other periods, assets or cost levels. Stress slices in the test period are few and non-contiguous. Named historical windows mostly lie in the training/validation periods and are **in-sample**.
* **Confidence is not a probability of profit.** It is temperature-scaled on validation proxy labels only and may drift out of calibration.
* The network can only be as conservative as the rule it imitates; rule blind spots are inherited. The BUY confidence filter is a soft preference, not a safety control.
* Gate thresholds are static; a mis-set gate can also *hurt* performance (compare `model_gated` vs `model_ungated`).

### Meta-RL integration contract
* **Load:** `ConservativePolicy.load("conservative_policy.pt", device)`; verify `feature_schema.json` SHA-256 equals the Meta-RL's expected schema hash and that actions use `0=HOLD, 1=BUY, 2=SELL, 3=EXIT`.
* **Input:** RAW float32 window `(20, F)` or `(B, 20, F)` in `feature_order`. Account-state/trader features must be computed by the caller exactly as defined in the schema. NaN/inf input is rejected.
* **Output of `predict(state)`:** `action_probabilities` (HOLD/BUY/SELL/EXIT), `action`, `action_name`, `raw_model_action`, `confidence`, `confidence_entropy`, `entry_filtered_to_hold`, `is_proposal_only = True`.
* **Authority:** this policy only *proposes*. Every action, including Meta-RL's final choice, must pass the independent deterministic risk controller, which may force EXIT or block BUY. Do not bypass or learn around it.
* **Use of confidence:** a relative signal calibrated to proxy labels; do not read it as the likelihood of a profitable or safe trade.
* **Scope:** research and paper trading only.